# Visão de MOB -- taxa pontual, por safra e por foto, e heatmap com slider de limiar

Três gráficos, todos usando a mesma definição: **taxa PONTUAL** de atraso --
"nesse MOB específico, quantos contratos estão em atraso agora, dividido
pelo total que existe nesse MOB agora" -- **não** é a taxa acumulada
("ever") que `construir_curvas_de_safra` usa. Um contrato que curou volta
a contar como bom; um contrato que quitou simplesmente sai do denominador
dali em diante (ele já não existe mais no painel).

**Pré-requisito:** assume que `df_painel_confiavel` já existe na sessão
(vem do notebook principal). Usa Plotly pela interatividade do slider no
gráfico 3 -- `%pip install plotly` antes, se o cluster não tiver.


In [ ]:
from pyspark.sql import functions as F
import pandas as pd
import plotly.graph_objects as go


## Preparação em Spark

Duas funções só, cada uma alimentando o que precisa:

- **`preparar_taxa_por_safra_mob_spark`** -- agrupa por `(safra, months_on_book)`,
  usando a definição já padrão do pipeline (`atraso_bucket == "90+"`).
  Alimenta o **Gráfico 1**.
- **`preparar_taxa_por_ref_month_mob_spark`** -- agrupa por
  `(ref_month, months_on_book)`, calculando a taxa para **vários limiares
  de dias de atraso de uma vez** (uma passada só pela base, não uma por
  limiar). Alimenta os **Gráficos 2 e 3**.


In [ ]:
def preparar_taxa_por_safra_mob_spark(df_painel, mob_maximo: int = 24) -> pd.DataFrame:
    """
    Taxa PONTUAL de atraso 90+ por (safra, months_on_book) -- "nesse MOB,
    dessa safra, quantos % estao em atraso 90+ agora" (nao acumulado).
    """
    filtrado = df_painel.filter(F.col("months_on_book") <= mob_maximo)
    resultado = (
        filtrado.withColumn("_ruim", (F.col("atraso_bucket") == "90+").cast("double"))
        .groupBy("safra", "months_on_book")
        .agg(F.avg("_ruim").alias("taxa_atraso90"), F.count(F.lit(1)).alias("n_contratos"))
        .orderBy("safra", "months_on_book")
    )
    return resultado.toPandas()


def preparar_taxa_por_ref_month_mob_spark(df_painel, limiares=(15, 30, 45, 60, 90, 120, 180),
                                             mob_maximo: int = 24) -> pd.DataFrame:
    """
    Taxa PONTUAL de atraso por (ref_month, months_on_book), para VARIOS
    limiares de dias de uma vez (uma agregacao so, nao uma por limiar) --
    usa `dias_atraso` direto (continuo), nao `atraso_bucket` (que so tem
    5 faixas fixas, sem granularidade pros limiares intermediarios tipo
    15/45/120/180).
    """
    filtrado = df_painel.filter(F.col("months_on_book") <= mob_maximo)
    agregacoes = [F.avg((F.col("dias_atraso") >= lim).cast("double")).alias(f"taxa_{lim}") for lim in limiares]
    agregacoes.append(F.count(F.lit(1)).alias("n_contratos"))
    resultado = filtrado.groupBy("ref_month", "months_on_book").agg(*agregacoes).orderBy("ref_month", "months_on_book")
    return resultado.toPandas()


## Gráfico 1 -- taxa pontual por safra, uma linha por MOB fixo

Compara safras **na mesma idade** -- cada linha é um MOB, o eixo X é a
safra. Essa é a comparação "justa" (mesma idade, safras diferentes).


In [ ]:
def plot_taxa_por_safra(tabela: pd.DataFrame, mobs_linha=(3, 6, 12)) -> go.Figure:
    fig = go.Figure()
    for mob in mobs_linha:
        sub = tabela[tabela["months_on_book"] == mob].sort_values("safra")
        fig.add_trace(go.Scatter(x=sub["safra"], y=sub["taxa_atraso90"], mode="lines+markers", name=f"MOB {mob}"))
    fig.update_layout(
        title="Taxa pontual de atraso 90+ por safra, em MOBs fixos",
        xaxis_title="Safra (originação)", yaxis_title="% em atraso 90+ (pontual)", height=420,
    )
    return fig


## Gráfico 2 -- mesma taxa, eixo X = mês de referência (foto)

Mesmas linhas (por MOB fixo), só que olhando pelo calendário -- **cada
ponto de uma linha é uma safra diferente** (por causa de
`safra = ref_month - MOB`). Útil pra ver "o que uma foto específica
mostra", não "como uma safra evolui".


In [ ]:
def plot_taxa_por_ref_month(tabela: pd.DataFrame, mobs_linha=(3, 6, 12)) -> go.Figure:
    fig = go.Figure()
    for mob in mobs_linha:
        sub = tabela[tabela["months_on_book"] == mob].sort_values("ref_month")
        fig.add_trace(go.Scatter(x=sub["ref_month"], y=sub["taxa_90"], mode="lines+markers", name=f"MOB {mob}"))
    fig.update_layout(
        title="Taxa pontual de atraso 90+ por mês de referência, em MOBs fixos",
        xaxis_title="Mês de referência (foto)", yaxis_title="% em atraso 90+ (pontual)", height=420,
    )
    return fig


## Gráfico 3 -- heatmap `ref_month` × MOB, com slider de limiar de atraso

Cor = taxa pontual, no limiar de dias que o slider está selecionando.
Células com menos de `n_minimo` contratos ficam **cinza** (duas camadas
de heatmap sobrepostas: uma cinza fixa por baixo, a colorida por cima com
`NaN` nas células de pouco dado -- onde vira `NaN`, a cinza de baixo
aparece).


In [ ]:
def plot_heatmap_limiar_atraso(tabela: pd.DataFrame, limiares=(15, 30, 45, 60, 90, 120, 180), n_minimo: int = 200) -> go.Figure:
    todos_ref_month = sorted(tabela["ref_month"].unique())
    todos_mob = sorted(tabela["months_on_book"].unique())
    ref_month_str = [pd.Timestamp(d).strftime("%Y-%m") for d in todos_ref_month]

    def _matriz(lim):
        piv_taxa = tabela.pivot(index="ref_month", columns="months_on_book", values=f"taxa_{lim}").reindex(index=todos_ref_month, columns=todos_mob)
        piv_n = tabela.pivot(index="ref_month", columns="months_on_book", values="n_contratos").reindex(index=todos_ref_month, columns=todos_mob)
        z = piv_taxa.values.copy()
        z[(piv_n.values < n_minimo) | pd.isna(piv_n.values)] = float("nan")
        return z

    z_fundo = [[0] * len(todos_mob) for _ in todos_ref_month]

    fig = go.Figure()
    fig.add_trace(go.Heatmap(z=z_fundo, x=todos_mob, y=ref_month_str,
                               colorscale=[[0, "#D9D9D9"], [1, "#D9D9D9"]], showscale=False, hoverinfo="skip"))
    fig.add_trace(go.Heatmap(z=_matriz(limiares[len(limiares) // 2]), x=todos_mob, y=ref_month_str,
                               colorscale="Reds", colorbar=dict(title="Taxa", tickformat=".0%"),
                               hovertemplate="Foto: %{y}<br>MOB: %{x}<br>Taxa: %{z:.2%}<extra></extra>"))

    fig.frames = [go.Frame(data=[go.Heatmap(z=z_fundo), go.Heatmap(z=_matriz(lim))], name=str(lim)) for lim in limiares]
    fig.update_layout(
        title="Taxa pontual por ref_month × MOB, por limiar de atraso (cinza = poucos contratos)",
        xaxis_title="Months on Book", yaxis_title="Mês de referência (foto)", height=650,
        sliders=[{
            "steps": [{"args": [[str(lim)], {"frame": {"duration": 0}, "mode": "immediate"}],
                        "label": f"{lim}+ dias", "method": "animate"} for lim in limiares],
            "currentvalue": {"prefix": "Limiar de atraso: "},
        }],
    )
    return fig


## Como usar

```python
tabela_safra = preparar_taxa_por_safra_mob_spark(df_painel_confiavel)
fig1 = plot_taxa_por_safra(tabela_safra, mobs_linha=(3, 6, 12))
fig1.show()

tabela_ref_month = preparar_taxa_por_ref_month_mob_spark(df_painel_confiavel)
fig2 = plot_taxa_por_ref_month(tabela_ref_month, mobs_linha=(3, 6, 12))
fig2.show()

fig3 = plot_heatmap_limiar_atraso(tabela_ref_month, n_minimo=200)
fig3.show()
```

Repara que `tabela_ref_month` alimenta os **Gráficos 2 e 3** -- uma
preparação Spark só, reaproveitada nos dois.

---
**Nota de transparência:** este notebook não foi executado contra Spark
real nesta sessão (ambiente sem PySpark disponível aqui) -- a lógica
segue exatamente o mesmo padrão já validado em dezenas de outras funções
do notebook principal (`groupBy`+`agg`, `.toPandas()` só na fatia já
pequena), e a mecânica do heatmap com slider foi validada com dado
sintético antes desta entrega. Vale rodar com atenção na primeira vez.
